### **Semana 3 - Prompting, structured generation, validación y context engineering con un LLM real**

#### **Pregunta experimental**

> ¿Añadir contexto conflictivo modifica la corrección semántica de un LLM cuando el modelo, el prompt, el contrato de salida y el decoding permanecen fijos?

Este cuaderno reemplaza el surrogate léxico por un **modelo generativo real**:

```text
input + contexto
      |
      v
Qwen2.5-0.5B-Instruct
      |
      v
texto generado
      |
      +-> parseo JSON exacto
      |
      +-> recuperación JSON opcional
      |
      v
JSON Schema
      |
      v
validación semántica
      |
      v
métricas + análisis de errores
```

El estándar experimental sigue siendo:

```text
pregunta -> hipótesis -> baseline -> una modificación -> métrica -> resultado -> limitación -> conclusión
```

#### Qué cambia respecto de un cuaderno controlado

```text
controlled_generate(...)
        |
        v
generate_with_llm(...)
```

No cambia:

```text
benchmark
prompt base
JSON Schema
condiciones de contexto
métricas
reglas de evaluación
```

El objetivo de Semana 3 continúa siendo **la interfaz entre generación probabilística y software verificable**, no aprender la API de Transformers.

#### **1. Modelo canónico**

Se utiliza:

```text
Qwen/Qwen2.5-0.5B-Instruct
```

Razones para esta semana:

- es un modelo *instruction-tuned* pequeño,
- puede ejecutarse localmente mediante `transformers`,
- soporta español,
- su ficha técnica destaca generación de datos estructurados y JSON,
- permite estudiar el comportamiento de un LLM real sin introducir una API comercial.

Modelo alternativo para una extensión:

```text
Qwen/Qwen2.5-1.5B-Instruct
```

El segundo modelo **no forma parte de la ejecución canónica del lunes**.

Referencias:

```text
https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct
https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
```

#### **2. Dos escalas de ejecución**

Para no confundir rigor con cantidad de cómputo:

```text
CLASSROOM_MODE = True
```

selecciona 12 casos balanceados:

```text
3 security
3 network
3 software
3 other
```

con al menos un caso ambiguo por categoría.

Entonces:

```text
12 casos x 3 condiciones = 36 generaciones
```

La extensión:

```text
CLASSROOM_MODE = False
```

ejecuta los 60 casos:

```text
60 x 3 = 180 generaciones
```

El protocolo es idéntico, solo cambia el tamaño de la muestra.

In [ ]:
from __future__ import annotations

import json
import hashlib
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from jsonschema import Draft202012Validator
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

CLASSROOM_MODE = True
CASES_PER_CATEGORY_CLASSROOM = 3

DO_SAMPLE = False
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 96

# En greedy una sola repetición es suficiente.
N_REPEATS = 1 if not DO_SAMPLE else 3

# True = reutiliza resultados ya guardados si existen.
REUSE_RESULTS = True

# La ejecución real está activada por defecto.
# Para validar el notebook sin descargar el modelo:
#   CC0F4_RUN_REAL_LLM=0 jupyter nbconvert ...
RUN_REAL_MODEL = os.environ.get("CC0F4_RUN_REAL_LLM", "1") == "1"

print("MODEL_ID:", MODEL_ID)
print("CLASSROOM_MODE:", CLASSROOM_MODE)
print("DO_SAMPLE:", DO_SAMPLE)
print("N_REPEATS:", N_REPEATS)
print("RUN_REAL_MODEL:", RUN_REAL_MODEL)
print("CUDA disponible:", torch.cuda.is_available())

#### **3. Contrato de salida**

El modelo debe producir exclusivamente:

```json
{
  "category": "network",
  "severity": 3,
  "summary": "Descripción breve"
}
```

`category` puede ser:

```text
security
network
software
other
unknown
```

`unknown` representa abstención explícita.

No incluimos un campo llamado `confidence`, porque un número generado por el modelo no sería automáticamente una probabilidad calibrada.

In [ ]:
INCIDENT_SCHEMA: Dict[str, Any] = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "security",
                "network",
                "software",
                "other",
                "unknown",
            ],
        },
        "severity": {
            "type": "integer",
            "minimum": 0,
            "maximum": 5,
        },
        "summary": {
            "type": "string",
            "minLength": 5,
            "maxLength": 180,
        },
    },
    "required": ["category", "severity", "summary"],
    "additionalProperties": False,
}

validator = Draft202012Validator(INCIDENT_SCHEMA)

CATEGORY_TO_SEVERITY = {
    "security": 5,
    "network": 3,
    "software": 2,
    "other": 1,
    "unknown": 0,
}

print(json.dumps(INCIDENT_SCHEMA, indent=2, ensure_ascii=False))

In [ ]:
# Universo explícito de etiquetas.
# `unknown` e `invalid` son resultados del sistema, no clases gold.
GOLD_LABELS = [
    "security",
    "network",
    "software",
    "other",
]

PREDICTION_LABELS = [
    *GOLD_LABELS,
    "unknown",
]

MATRIX_LABELS = [
    *PREDICTION_LABELS,
    "invalid",
]

_KNOWN_CATEGORIES = set(PREDICTION_LABELS)


def normalize_category_for_metrics(value: Any) -> str:
    """Colapsa un valor ausente o fuera del enum a 'invalid'."""
    if pd.isna(value):
        return "invalid"

    value = str(value)

    if value in _KNOWN_CATEGORIES:
        return value

    return "invalid"

#### **4. Prompt como especificación**

Se mantiene constante durante todo el experimento.

La variable experimental **no es el prompt**.

La única modificación principal es:

```text
relevant_context
        ->
relevant_context + conflicting_context
```

La condición `neutral` es un control negativo para distinguir:

```text
más contexto
```

de:

```text
contexto semánticamente conflictivo
```

In [ ]:
SYSTEM_PROMPT = """Eres un clasificador de incidentes.
Debes obedecer estrictamente el contrato de salida.
No escribas explicaciones fuera del objeto JSON.
"""

OUTPUT_CONTRACT = """Devuelve SOLO un objeto JSON válido con:
- category: security | network | software | other | unknown
- severity: entero 0..5
- summary: resumen breve en español

Reglas:
- security -> severity 5
- network -> severity 3
- software -> severity 2
- other -> severity 1
- unknown -> severity 0
- usa unknown solo si no puedes decidir una categoría única
- no agregues campos
- no uses Markdown
"""


def build_user_prompt(
    input_text: str,
    context: str,
) -> str:
    return f"""TAREA
Clasifica el incidente descrito en ENTRADA usando la información disponible.

ENTRADA
{input_text}

CONTEXTO
{context}

CONTRATO
{OUTPUT_CONTRACT}
"""


print(
    build_user_prompt(
        "Las consultas DNS fallan en el laboratorio.",
        "Los problemas de infraestructura de comunicaciones se atienden como red.",
    )
)

#### **5. Benchmark**

Se reutiliza exactamente el benchmark controlado de Semana 3.

Tiene:

```text
60 casos
15 por categoría
12 ambiguos
```

El benchmark no se modifica después de observar los resultados del LLM.

In [ ]:
def resolve_benchmark_path() -> Path:
    filename = "benchmark_incidentes_semana3.csv"

    explicit = os.environ.get("CC0F4_BENCHMARK")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"No existe CC0F4_BENCHMARK={path}")

    cwd = Path.cwd().resolve()

    candidates = []
    for root in [cwd, *cwd.parents]:
        candidates.extend(
            [
                root / "Semana3" / "datos" / filename,
                root / "datos" / filename,
                root / filename,
            ]
        )

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)

        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        "No se encontró benchmark_incidentes_semana3.csv. "
        "Define CC0F4_BENCHMARK si ejecutas desde otra ubicación."
    )


BENCHMARK_PATH = resolve_benchmark_path()
benchmark = pd.read_csv(BENCHMARK_PATH)

print("Benchmark:", BENCHMARK_PATH)
print("N =", len(benchmark))
print()
print(benchmark["gold_category"].value_counts())
print()
print(benchmark["difficulty"].value_counts())


def validate_benchmark(data: pd.DataFrame) -> None:
    required_columns = {
        "id",
        "input",
        "gold_category",
        "gold_severity",
        "difficulty",
        "relevant_context",
        "neutral_context",
        "distractor_context",
    }

    missing = required_columns - set(data.columns)
    if missing:
        raise ValueError(
            "Faltan columnas requeridas en el benchmark: "
            f"{sorted(missing)}"
        )

    categories = set(data["gold_category"].dropna().unique())
    expected_categories = set(GOLD_LABELS)

    if categories != expected_categories:
        raise ValueError(
            "Las categorías gold no coinciden con las esperadas. "
            f"Encontradas={sorted(categories)}, "
            f"esperadas={sorted(expected_categories)}"
        )

    difficulties = set(data["difficulty"].dropna().unique())
    if not difficulties.issubset({"clear", "ambiguous"}):
        raise ValueError(
            "difficulty contiene valores no reconocidos: "
            f"{sorted(difficulties)}"
        )

    if data["id"].duplicated().any():
        duplicated = data.loc[
            data["id"].duplicated(keep=False),
            "id",
        ].tolist()
        raise ValueError(
            f"El benchmark contiene ids duplicados: {duplicated}"
        )


validate_benchmark(benchmark)
print("Integridad del benchmark: OK")

In [ ]:
def build_classroom_subset(
    data: pd.DataFrame,
    cases_per_category: int = 3,
    seed: int = 42,
) -> pd.DataFrame:
    if cases_per_category < 2:
        raise ValueError(
            "cases_per_category debe ser >= 2 para seleccionar "
            "al menos un caso ambiguous y un caso clear por categoría."
        )

    selected = []

    for category in GOLD_LABELS:
        group = data[data["gold_category"] == category]

        ambiguous_pool = group[
            group["difficulty"] == "ambiguous"
        ]
        clear_pool = group[
            group["difficulty"] == "clear"
        ]

        required_ambiguous = 1
        required_clear = (
            cases_per_category - required_ambiguous
        )

        if len(ambiguous_pool) < required_ambiguous:
            raise ValueError(
                f"La categoría '{category}' requiere al menos "
                f"{required_ambiguous} caso ambiguous, "
                f"pero hay {len(ambiguous_pool)}."
            )

        if len(clear_pool) < required_clear:
            raise ValueError(
                f"La categoría '{category}' requiere al menos "
                f"{required_clear} casos clear, "
                f"pero hay {len(clear_pool)}."
            )

        ambiguous = ambiguous_pool.sample(
            n=required_ambiguous,
            random_state=seed,
        )

        clear = clear_pool.sample(
            n=required_clear,
            random_state=seed,
        )

        selected.append(
            pd.concat([clear, ambiguous])
        )

    subset = (
        pd.concat(selected)
        .sort_values(
            ["gold_category", "difficulty", "id"]
        )
        .reset_index(drop=True)
    )

    expected_size = (
        len(GOLD_LABELS) * cases_per_category
    )

    if len(subset) != expected_size:
        raise AssertionError(
            "El subset no tiene el tamaño esperado: "
            f"{len(subset)} != {expected_size}"
        )

    return subset


experiment_cases = (
    build_classroom_subset(
        benchmark,
        cases_per_category=CASES_PER_CATEGORY_CLASSROOM,
        seed=SEED,
    )
    if CLASSROOM_MODE
    else benchmark.copy()
)

print("Casos del experimento:", len(experiment_cases))
print()
print(
    experiment_cases
    .groupby(["gold_category", "difficulty"])
    .size()
)

#### **6. Condiciones de contexto**

Se utilizan exactamente los mismos casos en las tres condiciones:

```text
baseline:
    relevant_context

neutral:
    relevant_context + neutral_context

conflicting:
    relevant_context + distractor_context
```

La comparación científica principal es:

```text
baseline vs conflicting
```

In [ ]:
CONDITIONS = {
    "baseline": lambda row: row["relevant_context"],
    "neutral": lambda row: (
        row["relevant_context"]
        + "\n"
        + row["neutral_context"]
    ),
    "conflicting": lambda row: (
        row["relevant_context"]
        + "\n"
        + row["distractor_context"]
    ),
}

#### **7. Carga del modelo real**

La implementación utiliza el `chat_template` oficial del tokenizer.

El modelo se carga una sola vez.

No usamos `pipeline()` porque queremos que queden visibles:

```text
messages
tokenización
chat template
generate()
decoding
```

Esto conecta directamente con lo estudiado en Semana 2.

In [ ]:
tokenizer = None
model = None

if RUN_REAL_MODEL:
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Transformers no está instalado. "
            "Ejecuta el entorno global del curso con `make install-cpu` "
            "o `make install-gpu`."
        ) from exc

    model_revision = os.environ.get("CC0F4_MODEL_REVISION") or None

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=model_revision,
    )

    # `dtype` es la interfaz actual, el fallback mantiene compatibilidad
    # con versiones que todavía esperan `torch_dtype`.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            revision=model_revision,
            dtype="auto",
            device_map="auto",
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            revision=model_revision,
            torch_dtype="auto",
            device_map="auto",
        )

    model.eval()

    print("Modelo cargado:", MODEL_ID)
    print("Device principal:", model.device)
else:
    print("Carga del modelo omitida por CC0F4_RUN_REAL_LLM=0.")

#### **8. Generación**

`generate_with_llm()` devuelve tanto la respuesta textual como metadatos de ejecución.

Con greedy:

```text
DO_SAMPLE = False
```

la ejecución es determinista para un modelo/runtime dados.

Si luego se activa:

```text
DO_SAMPLE = True
```

se recomienda:

```text
N_REPEATS = 3
```

para estudiar variabilidad entre generaciones.

In [ ]:
def generate_with_llm(
    input_text: str,
    context: str,
    seed: int,
) -> Dict[str, Any]:
    if tokenizer is None or model is None:
        raise RuntimeError("El modelo real no está cargado.")

    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_prompt(input_text, context),
        },
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
    }

    if DO_SAMPLE:
        generation_kwargs.update(
            {
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
            }
        )

    start = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            **generation_kwargs,
        )

    elapsed_ms = (time.perf_counter() - start) * 1000.0

    prompt_tokens = model_inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0, prompt_tokens:]

    raw_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "raw_text": raw_text,
        "prompt_tokens": int(prompt_tokens),
        "generated_tokens": int(generated_ids.shape[-1]),
        "latency_ms": float(elapsed_ms),
    }

#### **9. Parseo: exacto frente a recuperación**

Con un LLM real aparecen dos preguntas distintas:

```text
¿el modelo devolvió JSON exacto?
```

y:

```text
¿podemos recuperar un objeto JSON de una respuesta con texto extra?
```

No debemos mezclarlas.

Métricas:

```text
raw_json_parse_rate
recovered_json_parse_rate
schema_valid_rate
```

La recuperación es **post-procesamiento**, no structured decoding.

In [ ]:
def parse_exact_json(raw_text: str) -> Optional[Dict[str, Any]]:
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        return None

    return data if isinstance(data, dict) else None


def recover_first_json_object(
    raw_text: str,
) -> Optional[Dict[str, Any]]:
    # Busca el primer objeto JSON decodificable.
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", raw_text):
        start = match.start()

        try:
            data, _ = decoder.raw_decode(raw_text[start:])
        except json.JSONDecodeError:
            continue

        if isinstance(data, dict):
            return data

    return None


def parse_generated_output(
    raw_text: str,
) -> Dict[str, Any]:
    exact = parse_exact_json(raw_text)

    if exact is not None:
        return {
            "parsed": exact,
            "parse_mode": "exact",
            "raw_json_valid": True,
            "recovered_json_valid": True,
        }

    recovered = recover_first_json_object(raw_text)

    return {
        "parsed": recovered,
        "parse_mode": (
            "recovered"
            if recovered is not None
            else "failed"
        ),
        "raw_json_valid": False,
        "recovered_json_valid": recovered is not None,
    }


# Tests locales del parser.
assert parse_generated_output(
    '{"category":"network","severity":3,"summary":"Falla DNS"}'
)["parse_mode"] == "exact"

assert parse_generated_output(
    'Respuesta: {"category":"network","severity":3,"summary":"Falla DNS"}'
)["parse_mode"] == "recovered"

assert parse_generated_output(
    "No puedo clasificar."
)["parse_mode"] == "failed"

print("Parser: OK")

#### **10. Validación estructural y semántica**

Una salida puede ser:

```text
parseable
pero no schema-valid
```

o:

```text
schema-valid
pero semánticamente incorrecta
```

Para el benchmark:

```text
semantic_correct
=
predicted_category == gold_category
```

`unknown` cuenta como abstención y como respuesta incorrecta, porque todos los casos tienen una etiqueta gold.

In [ ]:
def evaluate_parsed_output(
    parsed: Optional[Dict[str, Any]],
    gold_category: str,
) -> Dict[str, Any]:
    if parsed is None:
        return {
            "schema_valid": False,
            "pred_category": None,
            "pred_severity": None,
            "abstained": False,
            "category_correct": False,
            "severity_consistent": False,
        }

    schema_valid = validator.is_valid(parsed)

    if not schema_valid:
        return {
            "schema_valid": False,
            "pred_category": parsed.get("category"),
            "pred_severity": parsed.get("severity"),
            "abstained": parsed.get("category") == "unknown",
            "category_correct": False,
            "severity_consistent": False,
        }

    category = parsed["category"]
    severity = parsed["severity"]

    return {
        "schema_valid": True,
        "pred_category": category,
        "pred_severity": severity,
        "abstained": category == "unknown",
        "category_correct": category == gold_category,
        "severity_consistent": (
            severity == CATEGORY_TO_SEVERITY[category]
        ),
    }

#### **11. Ejecución experimental**

Cada fila registrada conserva:

```text
id
condition
repeat
gold
raw_text
parse_mode
schema_valid
prediction
correct
latency
tokens
```

Esto permite auditar posteriormente **qué generó realmente el modelo**.

Los resultados se guardan en:

```text
.build/semana3_llm/
```

para no tener que repetir generaciones durante una discusión posterior.

El nombre del CSV incluye una huella de la configuración
experimental. Si cambia el prompt, el benchmark, el modelo,
el decoding o los casos seleccionados, se genera una ruta
diferente y no se reutilizan silenciosamente resultados viejos.


In [ ]:
def safe_model_name(model_id: str) -> str:
    return model_id.replace("/", "__")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def resolve_model_revision() -> str:
    if model is None or tokenizer is None:
        return (
            os.environ.get("CC0F4_MODEL_REVISION")
            or "not-loaded"
        )

    config_revision = getattr(
        model.config,
        "_commit_hash",
        None,
    )

    tokenizer_revision = getattr(
        tokenizer,
        "init_kwargs",
        {},
    ).get("_commit_hash")

    return (
        config_revision
        or tokenizer_revision
        or os.environ.get("CC0F4_MODEL_REVISION")
        or "unresolved"
    )


BENCHMARK_SHA256 = sha256_file(
    BENCHMARK_PATH
)

RESOLVED_MODEL_REVISION = (
    resolve_model_revision()
)

EXPERIMENT_CONFIG = {
    "model_id": MODEL_ID,
    "model_revision": RESOLVED_MODEL_REVISION,
    "classroom_mode": CLASSROOM_MODE,
    "case_ids": experiment_cases["id"].tolist(),
    "do_sample": DO_SAMPLE,
    "temperature": (
        TEMPERATURE if DO_SAMPLE else None
    ),
    "top_p": TOP_P if DO_SAMPLE else None,
    "max_new_tokens": MAX_NEW_TOKENS,
    "n_repeats": N_REPEATS,
    "system_prompt": SYSTEM_PROMPT,
    "output_contract": OUTPUT_CONTRACT,
    "schema": INCIDENT_SCHEMA,
    "benchmark_sha256": BENCHMARK_SHA256,
    "conditions": {
        "baseline": "relevant_context",
        "neutral": (
            "relevant_context + neutral_context"
        ),
        "conflicting": (
            "relevant_context + distractor_context"
        ),
    },
}

EXPERIMENT_FINGERPRINT = hashlib.sha256(
    json.dumps(
        EXPERIMENT_CONFIG,
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()[:16]

BUILD_DIR = Path(".build") / "semana3_llm"
BUILD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

mode_name = (
    "classroom"
    if CLASSROOM_MODE
    else "full"
)

RESULTS_PATH = BUILD_DIR / (
    f"{safe_model_name(MODEL_ID)}"
    f"__{mode_name}"
    f"__cfg-{EXPERIMENT_FINGERPRINT}.csv"
)

METADATA_PATH = RESULTS_PATH.with_suffix(
    ".json"
)

METADATA_PATH.write_text(
    json.dumps(
        {
            "fingerprint": EXPERIMENT_FINGERPRINT,
            **EXPERIMENT_CONFIG,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Modelo:", MODEL_ID)
print(
    "Revisión resuelta:",
    RESOLVED_MODEL_REVISION,
)
print(
    "Benchmark SHA256:",
    BENCHMARK_SHA256[:16],
)
print(
    "Fingerprint:",
    EXPERIMENT_FINGERPRINT,
)
print("Resultados:", RESULTS_PATH)
print("Metadatos:", METADATA_PATH)

In [ ]:
def run_experiment(
    data: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    total = len(data) * len(CONDITIONS) * N_REPEATS
    step = 0

    for repeat in range(N_REPEATS):
        for _, case in data.iterrows():
            for condition_name, context_builder in CONDITIONS.items():
                step += 1
                context = context_builder(case)

                generation = generate_with_llm(
                    input_text=case["input"],
                    context=context,
                    seed=SEED + repeat,
                )

                parsed_info = parse_generated_output(
                    generation["raw_text"]
                )

                evaluation = evaluate_parsed_output(
                    parsed=parsed_info["parsed"],
                    gold_category=case["gold_category"],
                )

                rows.append(
                    {
                        "model_id": MODEL_ID,
                        "model_revision": RESOLVED_MODEL_REVISION,
                        "experiment_fingerprint": EXPERIMENT_FINGERPRINT,
                        "id": case["id"],
                        "condition": condition_name,
                        "repeat": repeat,
                        "difficulty": case["difficulty"],
                        "gold_category": case["gold_category"],
                        "input": case["input"],
                        "context": context,
                        **generation,
                        **parsed_info,
                        **evaluation,
                    }
                )

                print(
                    f"[{step:03d}/{total:03d}] "
                    f"{case['id']} "
                    f"{condition_name} "
                    f"-> {evaluation['pred_category']} "
                    f"schema={evaluation['schema_valid']}"
                )

    return pd.DataFrame(rows)


if RUN_REAL_MODEL:
    if REUSE_RESULTS and RESULTS_PATH.exists():
        results = pd.read_csv(RESULTS_PATH)

        required_fingerprint = {
            "experiment_fingerprint",
        }

        if not required_fingerprint.issubset(
            results.columns
        ):
            raise ValueError(
                "El CSV reutilizado no contiene "
                "experiment_fingerprint."
            )

        fingerprints = set(
            results[
                "experiment_fingerprint"
            ].dropna().astype(str)
        )

        if fingerprints != {
            EXPERIMENT_FINGERPRINT
        }:
            raise ValueError(
                "El CSV reutilizado pertenece a "
                "otra configuración experimental."
            )

        print(
            "Resultados reutilizados:",
            RESULTS_PATH,
        )
    else:
        results = run_experiment(experiment_cases)
        results.to_csv(
            RESULTS_PATH,
            index=False,
            encoding="utf-8",
        )
        print("Resultados guardados:", RESULTS_PATH)
else:
    results = pd.DataFrame()
    print(
        "Experimento omitido. "
        "Activa CC0F4_RUN_REAL_LLM=1 para generar resultados reales."
    )

#### **12. Métricas principales**

Para un LLM real ya no basta con `accuracy`.

Debemos separar:

```text
formato
    ->
raw_json_parse_rate

recuperabilidad
    ->
recovered_json_parse_rate

contrato
    ->
schema_valid_rate

semántica
    ->
category_accuracy
macro_f1

abstención
    ->
abstention_rate
```

También registramos latencia y tokens como métricas del sistema, aunque **no son la variable principal de Semana 3**.

In [ ]:
def summarize_results(
    data: pd.DataFrame,
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()

    rows = []

    for condition, group in data.groupby("condition"):
        rows.append(
            {
                "condition": condition,
                "n": len(group),
                "raw_json_parse_rate": group["raw_json_valid"].mean(),
                "recovered_json_parse_rate": group["recovered_json_valid"].mean(),
                "schema_valid_rate": group["schema_valid"].mean(),
                "category_accuracy": group["category_correct"].mean(),
                "abstention_rate": group["abstained"].mean(),
                "severity_consistency_rate": group["severity_consistent"].mean(),
                "mean_prompt_tokens": group["prompt_tokens"].mean(),
                "mean_generated_tokens": group["generated_tokens"].mean(),
                "mean_latency_ms": group["latency_ms"].mean(),
            }
        )

    return pd.DataFrame(rows).sort_values("condition")


summary = summarize_results(results)

if not summary.empty:
    print(summary.to_string(index=False))
else:
    print("No hay resultados reales todavía.")

#### **13. Macro-F1**

Para evitar que una categoría domine el resultado, reportamos también macro-F1.

Las salidas inválidas y `unknown` se conservan como etiquetas distintas, de modo que no desaparezcan de la evaluación.

In [ ]:
def compute_macro_f1_by_condition(
    data: pd.DataFrame,
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()

    rows = []

    for condition, group in data.groupby("condition"):
        predicted = group[
            "pred_category"
        ].map(normalize_category_for_metrics)

        rows.append(
            {
                "condition": condition,
                "macro_f1": f1_score(
                    group["gold_category"],
                    predicted,
                    labels=GOLD_LABELS,
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    return pd.DataFrame(rows)


macro_f1_table = compute_macro_f1_by_condition(results)

if not macro_f1_table.empty:
    print(macro_f1_table.to_string(index=False))

**Interpretación de Macro-F1.**

El promedio se calcula sobre las cuatro clases semánticas
`security`, `network`, `software` y `other`.

`unknown` e `invalid` no son clases objetivo del benchmark.
Cuando el modelo produce una de ellas, la clase gold
correspondiente recibe un falso negativo, por lo que el error
sí penaliza Macro-F1.

Estas salidas se reportan además de forma explícita mediante:

```text
abstention_rate
raw_json_parse_rate
schema_valid_rate
```

#### **14. Comparación pareada**

La unidad experimental es el **mismo caso** bajo contextos distintos.

Por eso interesa contar:

```text
baseline correcto -> conflicting incorrecto
```

y:

```text
baseline incorrecto -> conflicting correcto
```

Con greedy y `N_REPEATS=1`, el pareo es directo.

In [ ]:
def paired_flip_table(
    data: pd.DataFrame,
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()

    paired = (
        data[
            data["condition"].isin(["baseline", "conflicting"])
        ]
        .pivot(
            index=["id", "repeat"],
            columns="condition",
            values="category_correct",
        )
        .dropna()
        .astype(bool)
    )

    paired["baseline_to_error"] = (
        paired["baseline"]
        & ~paired["conflicting"]
    )

    paired["error_to_conflicting_correct"] = (
        ~paired["baseline"]
        & paired["conflicting"]
    )

    return paired


paired = paired_flip_table(results)

if not paired.empty:
    b = int(paired["baseline_to_error"].sum())
    c = int(paired["error_to_conflicting_correct"].sum())

    print("baseline correcto -> conflicting incorrecto:", b)
    print("baseline incorrecto -> conflicting correcto:", c)
    print("pares:", len(paired))

#### **15. Bootstrap pareado del delta de accuracy**

Definimos:

```text
delta = accuracy_baseline - accuracy_conflicting
```

El bootstrap re-muestrea pares completos.

En modo aula el intervalo puede ser amplio, eso es una **limitación esperada**, no un error del experimento.

In [ ]:
def aggregate_repeats_by_case(
    paired_table: pd.DataFrame,
) -> pd.DataFrame:
    if paired_table.empty:
        return pd.DataFrame(
            columns=["baseline", "conflicting"]
        )

    # La unidad experimental es el caso.
    # Si hay sampling, primero se promedian las repeticiones del mismo id.
    if isinstance(paired_table.index, pd.MultiIndex):
        return (
            paired_table[
                ["baseline", "conflicting"]
            ]
            .astype(float)
            .groupby(level="id")
            .mean()
        )

    return paired_table[
        ["baseline", "conflicting"]
    ].astype(float)


def paired_bootstrap_delta(
    case_level_table: pd.DataFrame,
    n_bootstrap: int = 5000,
    seed: int = 42,
) -> np.ndarray:
    if case_level_table.empty:
        return np.array([])

    rng = np.random.default_rng(seed)

    values = case_level_table[
        ["baseline", "conflicting"]
    ].to_numpy(dtype=float)

    n_cases = len(values)
    deltas = np.empty(
        n_bootstrap,
        dtype=float,
    )

    for i in range(n_bootstrap):
        indices = rng.integers(
            low=0,
            high=n_cases,
            size=n_cases,
        )

        sample = values[indices]

        deltas[i] = (
            sample[:, 0].mean()
            - sample[:, 1].mean()
        )

    return deltas


if not paired.empty:
    case_level_paired = aggregate_repeats_by_case(
        paired
    )

    observed_delta = (
        case_level_paired["baseline"].mean()
        - case_level_paired["conflicting"].mean()
    )

    bootstrap_deltas = paired_bootstrap_delta(
        case_level_paired
    )

    ci_low, ci_high = np.quantile(
        bootstrap_deltas,
        [0.025, 0.975],
    )

    print("Unidad de bootstrap: caso")
    print("Repeticiones por caso:", N_REPEATS)
    print(f"Delta observado: {observed_delta:.4f}")
    print(
        f"IC bootstrap 95%: "
        f"[{ci_low:.4f}, {ci_high:.4f}]"
    )

#### **16. Matriz de confusión del contexto conflictivo**

La matriz de confusión responde:

```text
¿qué categorías atraen los errores?
```

Esto es más informativo que limitarse a una única accuracy.

In [ ]:
if not results.empty:
    conflicting = results[
        results["condition"] == "conflicting"
    ].copy()

    conflicting["pred_for_cm"] = (
        conflicting["pred_category"]
        .map(normalize_category_for_metrics)
    )

    unexpected_mask = (
        conflicting["pred_category"].notna()
        & ~conflicting["pred_category"].isin(
            PREDICTION_LABELS
        )
    )

    unexpected_predictions = conflicting.loc[
        unexpected_mask,
        [
            "id",
            "repeat",
            "pred_category",
            "schema_valid",
            "raw_text",
        ],
    ].copy()

    print(
        "Predicciones fuera del enum:",
        len(unexpected_predictions),
    )

    if not unexpected_predictions.empty:
        display(unexpected_predictions)

    cm = confusion_matrix(
        conflicting["gold_category"],
        conflicting["pred_for_cm"],
        labels=MATRIX_LABELS,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"gold:{label}"
            for label in MATRIX_LABELS
        ],
        columns=[
            f"pred:{label}"
            for label in MATRIX_LABELS
        ],
    )

    # Ninguna observación puede desaparecer silenciosamente.
    assert int(cm.sum()) == len(conflicting)

    display(cm_df)

#### **17. Auditoría de errores**

No basta con preguntar:

```text
¿cuántos fallaron?
```

También debemos preguntar:

```text
¿qué texto generó el modelo?
¿falló el formato?
¿falló el schema?
¿cambió de categoría?
¿se abstuvo?
```

In [ ]:
if not results.empty and not paired.empty:
    flip_keys = (
        paired[
            paired["baseline_to_error"]
        ]
        .reset_index()[
            ["id", "repeat"]
        ]
    )

    if flip_keys.empty:
        print(
            "No hubo flips baseline correcto -> "
            "conflicting incorrecto."
        )
    else:
        audit = (
            results.merge(
                flip_keys,
                on=["id", "repeat"],
                how="inner",
                validate="many_to_one",
            )[
                [
                    "id",
                    "repeat",
                    "condition",
                    "difficulty",
                    "gold_category",
                    "pred_category",
                    "parse_mode",
                    "schema_valid",
                    "raw_text",
                ]
            ]
            .sort_values(
                ["id", "repeat", "condition"]
            )
        )

        print(
            "Pares (id, repeat) auditados:",
            len(flip_keys),
        )
        display(audit)

#### 18.** Qué significa `baseline ~= neutral > conflicting`**

Ese patrón sería evidencia compatible con:

```text
añadir longitud neutral
        ->
poco cambio

añadir información conflictiva
        ->
mayor degradación
```

Pero el cuaderno **no presupone** que ese patrón aparecerá.

Resultados igualmente válidos serían:

```text
baseline ~= neutral ~= conflicting
```

o incluso:

```text
conflicting > baseline
```

Si ocurren, deben reportarse.

No se modifican después:

```text
benchmark
pesos
prompts
contextos
métricas
```

para fabricar una diferencia deseada.

#### **19. Extensión con sampling**

La ejecución canónica usa greedy:

```text
do_sample = False
N_REPEATS = 1
```

Para estudiar estabilidad puede cambiarse a:

```text
DO_SAMPLE = True
N_REPEATS = 3
```

manteniendo:

```text
temperature
top_p
seed policy
```

fijos entre las tres condiciones.

Entonces puede añadirse:

```text
flip_rate
variance entre repeticiones
acuerdo intra-caso
```

La repetición tiene sentido cuando existe estocasticidad.

#### **20. Extensión con segundo modelo**

Una réplica natural utiliza:

```text
Qwen/Qwen2.5-1.5B-Instruct
```

sin cambiar ninguna otra parte del protocolo.

Entonces la nueva pregunta es:

> ¿La sensibilidad al contexto conflictivo depende del modelo?.

Comparación:

```text
0.5B:
baseline - conflicting = delta_0.5B

1.5B:
baseline - conflicting = delta_1.5B
```

Esta extensión ya se aproxima más a un pequeño estudio experimental que a una demostración didáctica.

### **21. Ejercicios de refuerzo**

Los siguientes ejercicios utilizan el mismo modelo, benchmark, contrato y funciones del cuaderno. El objetivo es comprobar que puedes modificar **una variable a la vez**, medir su efecto y justificar la conclusión.

#### **Ejercicio 1 - JSON válido no implica respuesta correcta**

Busca en `results` una generación que cumpla:

```text
schema_valid = True
category_correct = False
```

Muestre:

- `input`
- `condition`
- `gold_category`
- `pred_category`
- `raw_text`

Explica por qué el JSON Schema puede aceptar esa respuesta aunque la clasificación sea incorrecta.

**Pregunta:** ¿qué propiedad verifica el schema y qué propiedad debe verificarse mediante evaluación semántica?


#### **Ejercicio 2 - JSON exacto vs JSON recuperable**

Busca ejemplos correspondientes a:

```text
parse_mode = exact
parse_mode = recovered
parse_mode = failed
```

Si alguna categoría no aparece en la ejecución actual, construya manualmente una cadena de ejemplo y pásela por:

```python
parse_generated_output(...)
```

Explica la diferencia entre:

```text
JSON exacto
->
JSON recuperable
->
JSON Schema válido
```

**Pregunta:** ¿por qué recuperar un objeto JSON mediante post-procesamiento no equivale a utilizar constrained generation?.

#### **Ejercicio 3 - Prompt y contexto no son lo mismo**

Selecciona un caso del benchmark y muestra:

```text
input
relevant_context
neutral_context
distractor_context
```

Construye los tres prompts usados en:

```text
baseline
neutral
conflicting
```

Identifica qué parte permanece fija y qué parte constituye la variable experimental.

**Pregunta:** ¿por qué modificar también la instrucción principal impediría atribuir el cambio observado únicamente al contexto?.


#### Ejercicio 4 - Analizar un cambio de predicción

Identifica un caso donde:

```text
baseline
!=
conflicting
```

Compara ambas generaciones y completa:

```text
id:
gold_category:
baseline prediction:
conflicting prediction:
baseline schema_valid:
conflicting schema_valid:
```

Después propone una explicación basada en la información añadida al contexto.

No es suficiente afirmar que "el modelo se confundió". Debes señalar qué evidencia del contexto podría haber influido en la nueva clasificación.


#### **Ejercicio 5 - El control neutral**

Calcula para las tres condiciones:

```text
category_accuracy
schema_valid_rate
abstention_rate
```

Compara:

```text
baseline <-> neutral
baseline <-> conflicting
```

Responde:

1. ¿Agregar más texto neutral cambia el resultado?
2. ¿Agregar información conflictiva cambia el resultado?
3. ¿Los datos permiten distinguir entre efecto de longitud y efecto semántico?

La conclusión debe limitarse a lo que muestran los datos de esta ejecución.


#### **Ejercicio 6 - Métrica agregada vs análisis de errores**

Supongamos que dos condiciones tienen la misma `category_accuracy`.

¿Significa eso que producen exactamente los mismos errores?

Utiliza las predicciones para encontrar, si existen:

```text
casos correctos en ambas condiciones
casos incorrectos en ambas condiciones
baseline correcto -> conflicting incorrecto
baseline incorrecto -> conflicting correcto
```

**Pregunta:** ¿por qué una única métrica agregada puede ocultar comportamientos diferentes?.


#### **Ejercicio 7 - Abstención**

Busca las predicciones:

```text
pred_category = unknown
```

Si no existen en la ejecución actual, explica igualmente cómo se evalúan en el cuaderno.

Responde:

1. ¿`unknown` pertenece a las clases gold?
2. ¿Debe contarse como respuesta correcta?
3. ¿Por qué puede ser preferible permitir abstención en lugar de forzar siempre una categoría?
4. ¿Qué métrica del cuaderno permite observar este comportamiento?.


#### **Ejercicio 8 - Diseñar una nueva hipótesis**

Propone una modificación experimental que pueda estudiarse sin cambiar simultáneamente varias variables.

Ejemplos posibles:

```text
greedy -> sampling
Qwen2.5-0.5B -> Qwen2.5-1.5B
prompt actual -> prompt con una regla adicional
contexto corto -> contexto más largo
```

Completa antes de ejecutar:

```text
Pregunta:
Hipótesis:
Baseline:
Única modificación:
Métrica principal:
Variables que deben permanecer fijas:
Resultado esperado:
```

Después de ejecutar completa:

```text
Resultado:
Limitación:
Conclusión:
```

No modifiques la hipótesis después de observar los resultados.


#### **Ejercicio 9 - Conclusión científica**

A partir de los resultados obtenidos en el cuaderno, escribe una conclusión de máximo cinco líneas que contenga explícitamente:

```text
qué se comparó
qué métrica cambió
qué métrica no cambió
qué limitación existe
qué NO puede generalizarse
```

Evita conclusiones como:

> Más contexto empeora los LLM.

Prefiere afirmaciones del tipo:

> Bajo este modelo, benchmark y protocolo experimental, se observó...


In [ ]:
### Tus respuestas

#### **22. Puente a la Semana 4**

En este cuaderno el contexto sigue estando seleccionado manualmente.

Semana 4 preguntará:

```text
¿cómo recuperar automáticamente el contexto relevante? -> embeddings -> similitud  -> chunking -> dense retrieval -> FAISS
```